# 02 Calibration Loader

This notebook loads sparse calibration CSV files. Calibration files may pre-date 2026-05-29 because they are treated as calibration history, not measurement data.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

DATA_DIR = Path("..") / "test_data"
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

print("pandas", pd.__version__)
print("calibration directory", DATA_DIR.resolve())

In [ ]:
def calibration_file_date(path: Path) -> str:
    match = re.fullmatch(r"calibration_(\d{8})\.csv", path.name)
    return match.group(1) if match else ""


def load_calibration_files(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    files = sorted(path for path in data_dir.glob("calibration_*.csv") if calibration_file_date(path))
    if not files:
        return pd.DataFrame()
    frames = []
    for path in files:
        frame = pd.read_csv(
            path,
            na_values=["", "nan", "NaN", "NAN"],
            keep_default_na=True,
        )
        frame.insert(0, "source_file", path.name)
        frame.insert(1, "source_date", calibration_file_date(path))
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


cal_raw = load_calibration_files()
assert not cal_raw.empty, "No calibration files found."
cal_raw["timestamp_utc"] = pd.to_datetime(cal_raw["timestamp_utc"], errors="coerce", utc=False)

print("Calibration rows:", len(cal_raw))
print("Calibration files:", sorted(cal_raw["source_file"].unique()))
cal_raw.head()

In [ ]:
VALUE_RE = re.compile(r"^(sen|ref)_(.+)_(\d+)_(zero|span|diff)$")
COEFF_RE = re.compile(r"^(.+)_(\d+)_(gain|offset|flag)$")


def tidy_calibration_values(cal_raw: pd.DataFrame) -> pd.DataFrame:
    id_cols = ["source_file", "source_date", "timestamp_utc"]
    records = []
    for _, row in cal_raw.iterrows():
        for col in cal_raw.columns:
            match = VALUE_RE.match(col)
            if not match:
                continue
            value = row[col]
            records.append({
                "source_file": row["source_file"],
                "source_date": row["source_date"],
                "timestamp_utc": row["timestamp_utc"],
                "data_type": match.group(1),
                "gas": match.group(2),
                "sensor": int(match.group(3)),
                "calibration_type": match.group(4),
                "value": value,
                "is_observed_in_row": pd.notna(value),
            })
    return pd.DataFrame.from_records(records)


def tidy_calibration_coefficients(cal_raw: pd.DataFrame) -> pd.DataFrame:
    records = []
    for _, row in cal_raw.iterrows():
        keyed = {}
        for col in cal_raw.columns:
            match = COEFF_RE.match(col)
            if not match:
                continue
            key = (match.group(1), int(match.group(2)))
            keyed.setdefault(key, {})[match.group(3)] = row[col]
        for (gas, sensor), values in keyed.items():
            records.append({
                "source_file": row["source_file"],
                "source_date": row["source_date"],
                "timestamp_utc": row["timestamp_utc"],
                "gas": gas,
                "sensor": sensor,
                "gain": values.get("gain"),
                "offset": values.get("offset"),
                "flag": values.get("flag"),
                "coefficients_observed_in_row": any(pd.notna(v) for v in values.values()),
            })
    return pd.DataFrame.from_records(records)


cal_values_tidy = tidy_calibration_values(cal_raw)
cal_coeffs_tidy = tidy_calibration_coefficients(cal_raw)

print("Tidy calibration value rows:", len(cal_values_tidy))
print("Tidy coefficient rows:", len(cal_coeffs_tidy))
display(cal_values_tidy.query("is_observed_in_row").head(30))
display(cal_coeffs_tidy.query("coefficients_observed_in_row").head(30))

In [ ]:
def forward_filled_calibration_state(cal_values_tidy: pd.DataFrame, cal_coeffs_tidy: pd.DataFrame) -> pd.DataFrame:
    value_state = (
        cal_values_tidy
        .sort_values(["gas", "sensor", "data_type", "calibration_type", "timestamp_utc", "source_file"])
        .copy()
    )
    value_state["value_ffill"] = value_state.groupby(
        ["gas", "sensor", "data_type", "calibration_type"], dropna=False
    )["value"].ffill()

    coeff_state = (
        cal_coeffs_tidy
        .sort_values(["gas", "sensor", "timestamp_utc", "source_file"])
        .copy()
    )
    for col in ["gain", "offset", "flag"]:
        coeff_state[f"{col}_ffill"] = coeff_state.groupby(["gas", "sensor"], dropna=False)[col].ffill()
    return value_state, coeff_state


cal_value_state, cal_coeff_state = forward_filled_calibration_state(cal_values_tidy, cal_coeffs_tidy)

print("Observed sparse value rows:")
display(cal_value_state.query("is_observed_in_row").head(30))

print("Forward-filled coefficient state:")
display(cal_coeff_state.head(30))

In [ ]:
latest_coefficients = (
    cal_coeff_state
    .sort_values(["gas", "sensor", "timestamp_utc", "source_file"])
    .groupby(["gas", "sensor"], as_index=False)
    .tail(1)
    [["gas", "sensor", "timestamp_utc", "gain_ffill", "offset_ffill", "flag_ffill"]]
    .sort_values(["gas", "sensor"])
)

latest_coefficients